In [6]:
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from bs4 import BeautifulSoup

LIST_URL = "https://youth.europa.eu/go-abroad/volunteering/opportunities_en"
BASE_URL = "https://youth.europa.eu"

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("window-size=1920x1080")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

print(f"Selenium ile ana ilan listesine bağlanılıyor: {LIST_URL}...")
hedef_proje_sayisi = 0

try:
    driver.get(LIST_URL)
    
    # 1. HEDEF PROJE SAYISINI ÖĞREN
    try:
        hedef_etiket_xpath = "//h5/span[contains(text(), 'opportunities found')]"
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, hedef_etiket_xpath))
        )
        hedef_metin = driver.find_element(By.XPATH, hedef_etiket_xpath).text
        match = re.search(r'(\d+)', hedef_metin)
        if match:
            hedef_proje_sayisi = int(match.group(1))
            print(f"Hedef proje sayısı bulundu: {hedef_proje_sayisi}")
        else:
            raise Exception("Hedef sayı metni bulundu ama sayı çekilemedi.")
            
    except Exception as e:
        print(f"HATA: Toplam proje sayısı bulunamadı: {e}")
        raise

    # 2. "Load more" butonu XPATH'i
    load_more_xpath = "//button[contains(text(), 'Load more')]"
    
    # 3. HİBRİT DÖNGÜ: Hedef sayıya ulaşana KADAR tıkla
    while len(driver.find_elements(By.CLASS_NAME, "card-item")) < hedef_proje_sayisi:
        try:
            mevcut_proje_sayisi = len(driver.find_elements(By.CLASS_NAME, "card-item"))
            
            if mevcut_proje_sayisi >= hedef_proje_sayisi:
                print(f"Tüm ilanlar yüklendi (Mevcut: {mevcut_proje_sayisi} / Hedef: {hedef_proje_sayisi}).")
                break

            load_more_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, load_more_xpath))
            )
            driver.execute_script("arguments[0].click();", load_more_button)
            
            print(f"-> 'Load more' tıklandı. Yüklenen: {mevcut_proje_sayisi}/{hedef_proje_sayisi}")
            time.sleep(2) 
            
        except (NoSuchElementException, TimeoutException):
            mevcut_proje_sayisi = len(driver.find_elements(By.CLASS_NAME, "card-item"))
            print(f"Tüm ilanlar yüklendi (Load more butonu kayboldu). Yüklenen: {mevcut_proje_sayisi}")
            break
        except Exception as e:
            print(f"Butona tıklarken beklenmedik bir hata oluştu: {e}")
            break

    # 4. Tüm ilanlar yüklendikten sonra, LİNKLERİ TOPLA
    print("Sayfanın tamamı yüklendi. Linkler toplanıyor...")
    html_source = driver.page_source
    soup = BeautifulSoup(html_source, "html.parser")
    ilan_bloklari = soup.find_all("div", class_="card-item")
    
    link_listesi = []
    for blok in ilan_bloklari:
        link_element = blok.find("div", class_="card-action").find("a")
        if link_element and link_element.has_attr('href'):
            link = BASE_URL + link_element['href']
            link_listesi.append(link)

    # 5. Linkleri dosyaya yaz
    with open("proje_linkleri.txt", "w") as f:
        for link in link_listesi:
            f.write(link + "\n")

    print(f"\n--- KOD 1 TAMAMLANDI ---")
    print(f"Toplam {len(link_listesi)} adet proje linki 'proje_linkleri.txt' dosyasına kaydedildi.")

except Exception as e:
    print(f"HATA: Selenium çalışırken ana bir sorun oluştu: {e}")

finally:
    driver.quit()
    print("Tarayıcı kapatıldı.")

Selenium ile ana ilan listesine bağlanılıyor: https://youth.europa.eu/go-abroad/volunteering/opportunities_en...
Hedef proje sayısı bulundu: 951
-> 'Load more' tıklandı. Yüklenen: 8/951
-> 'Load more' tıklandı. Yüklenen: 16/951
-> 'Load more' tıklandı. Yüklenen: 16/951
-> 'Load more' tıklandı. Yüklenen: 32/951
-> 'Load more' tıklandı. Yüklenen: 40/951
-> 'Load more' tıklandı. Yüklenen: 48/951
-> 'Load more' tıklandı. Yüklenen: 56/951
-> 'Load more' tıklandı. Yüklenen: 56/951
-> 'Load more' tıklandı. Yüklenen: 72/951
-> 'Load more' tıklandı. Yüklenen: 72/951
-> 'Load more' tıklandı. Yüklenen: 80/951
-> 'Load more' tıklandı. Yüklenen: 88/951
-> 'Load more' tıklandı. Yüklenen: 104/951
-> 'Load more' tıklandı. Yüklenen: 112/951
-> 'Load more' tıklandı. Yüklenen: 120/951
-> 'Load more' tıklandı. Yüklenen: 128/951
-> 'Load more' tıklandı. Yüklenen: 136/951
-> 'Load more' tıklandı. Yüklenen: 144/951
-> 'Load more' tıklandı. Yüklenen: 152/951
-> 'Load more' tıklandı. Yüklenen: 160/951
-> 'Load

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://youth.europa.eu"

# -------------------------------------------------------------------------
# DETAY SAYFASININ ANALİZ FONKSİYONU
# (429 Hatası Yönetimi ve Yeniden Deneme (Retry) Mantığı Eklendi)
# -------------------------------------------------------------------------
def extract_project_description_robust(detail_url, retries=1):
    """
    URL'ye gider, 429 hatası alırsa 10 saniye bekler ve tekrar dener.
    """
    try:
        page = requests.get(detail_url)
        page.raise_for_status() # Hata varsa (429 dahil) exception fırlat
        
        soup = BeautifulSoup(page.content, "html.parser")
        content_cards = soup.find_all("div", class_="card-content")
        full_description_text = ""
        
        for card in content_cards:
            h6_header = card.find("h6")
            if h6_header and "Description" in h6_header.text:
                all_paragraphs = card.find_all("p")
                for p in all_paragraphs:
                    full_description_text += p.text.strip() + "\n\n" 
                break 
        
        return full_description_text.strip() if full_description_text else "AÇIKLAMA BULUNAMADI"
            
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 429 and retries > 0:
            print("    -> 429 Hatası! Sunucu engelledi. 10 saniye bekleniyor ve tekrar denenecek...")
            time.sleep(10) # Engeli aşmak için UZUN bir bekleme
            # Kalan deneme hakkını bir azaltarak fonksiyonu tekrar çağır
            return extract_project_description_robust(detail_url, retries - 1)
        else:
            return f"HATA: {e.response.status_code} Hatası"
    except requests.exceptions.RequestException as e:
        return "HATA: Bağlantı Hatası"

# -------------------------------------------------------------------------
# ANA İŞLEM DÖNGÜSÜ (KOD 2)
# -------------------------------------------------------------------------
print("--- KOD 2 BAŞLADI: Detaylar Çekiliyor ---")

try:
    with open("proje_linkleri.txt", "r") as f:
        link_listesi = [line.strip() for line in f.readlines()]
    print(f"'{len(link_listesi)}' adet link dosyadan okundu.")
except FileNotFoundError:
    print("HATA: 'proje_linkleri.txt' dosyası bulunamadı. Lütfen önce Kod 1'i çalıştırın.")
    # raise komutu bu hücrenin çalışmasını durdurur
    raise

proje_verileri = []
basarisiz_linkler = 0

for i, link in enumerate(link_listesi):
    print(f"İşleniyor... ({i+1}/{len(link_listesi)}) - {link}")
    
    # Ana fonksiyonu çağır (1 yeniden deneme hakkı var)
    full_description = extract_project_description_robust(link, retries=1)

    if "HATA" in full_description or "BULUNAMADI" in full_description:
        print(f"    -> UYARI: Bu ilan atlanıyor. (Sebep: {full_description[:50]}...)")
        basarisiz_linkler += 1

    proje_verileri.append({
        "link": link,
        "tam_aciklama": full_description
    })
    
    # --- 429 ENGELLEYİCİ ---
    # Her bir istek arasında 1 saniye bekle. (Bu satır çok önemlidir)
    time.sleep(1) 

# 6. Veriyi DataFrame'e dönüştür ve kaydet
df_projeler = pd.DataFrame(proje_verileri)
# 'youth_portal_projeler.csv' dosyasındaki başlık vb. bilgilerle birleştirebiliriz
# ama şimdilik sadece açıklamaları kaydedelim:
df_projeler.to_csv("youth_portal_projeler_detayli.csv", index=False, encoding='utf-8-sig')

print(f"\n--- KOD 2 TAMAMLANDI ---")
print(f"Toplam {len(df_projeler)} proje ilanı işlendi.")
if basarisiz_linkler > 0:
    print(f"UYARI: {basarisiz_linkler} adet ilanın detay sayfası ÇEKİLEMEDİ.")
print("Tüm veriler 'youth_portal_projeler_detayli.csv' dosyasına kaydedildi.")

--- KOD 2 BAŞLADI: Detaylar Çekiliyor ---
'952' adet link dosyadan okundu.
İşleniyor... (1/952) - https://youth.europa.eu/solidarity/opportunity/49017_en
İşleniyor... (2/952) - https://youth.europa.eu/solidarity/opportunity/49016_en
İşleniyor... (3/952) - https://youth.europa.eu/solidarity/opportunity/49015_en
İşleniyor... (4/952) - https://youth.europa.eu/solidarity/opportunity/49014_en
İşleniyor... (5/952) - https://youth.europa.eu/solidarity/opportunity/49013_en
İşleniyor... (6/952) - https://youth.europa.eu/solidarity/opportunity/49012_en
İşleniyor... (7/952) - https://youth.europa.eu/solidarity/opportunity/49011_en
İşleniyor... (8/952) - https://youth.europa.eu/solidarity/opportunity/49010_en
İşleniyor... (9/952) - https://youth.europa.eu/solidarity/opportunity/49011_en
İşleniyor... (10/952) - https://youth.europa.eu/solidarity/opportunity/49010_en
İşleniyor... (11/952) - https://youth.europa.eu/solidarity/opportunity/49009_en
İşleniyor... (12/952) - https://youth.europa.eu/solida

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# -------------------------------------------------------------------------
# ADIM 3: EUROPASS CV PARSER (SELENIUM İLE GÜNCELLENDİ)
# -------------------------------------------------------------------------

CV_URL = "https://europa.eu/europass/eportfolio/api/eprofile/shared-profile/aziz+muammer-%C3%A7etiner/7166a565-f20c-4c5c-83a4-00a4db71fd09?view=html"

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("window-size=1920x1080")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

print(f"İP 1 - Adım 3 Başladı: Europass CV linki Selenium ile çekiliyor...\nURL: {CV_URL}")

cv_text_list = []
cv_beceri_listesi = [] 

try:
    driver.get(CV_URL)
    
    # Sayfanın dinamik olarak yüklenmesini bekle (Skills başlığı var olana kadar)
    skills_header_xpath = "//h2[contains(text(), 'Skills')]"
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.XPATH, skills_header_xpath))
    )
    print("Europass profili (JavaScript) yüklendi. Veri ayrıştırılıyor...")
    
    html_source = driver.page_source
    soup = BeautifulSoup(html_source, "html.parser")

    # --- 1. Eğitim (Education and training) Bölümünü Çek ---
    edu_header = soup.find("h2", string=lambda t: t and "Education and training" in t)
    if edu_header:
        current_tag = edu_header.find_next_sibling()
        while current_tag and current_tag.name != 'h2':
            if current_tag.name == 'p' or current_tag.name == 'h4':
                cv_text_list.append(current_tag.text.strip())
            current_tag = current_tag.find_next_sibling()
            
    # --- 2. İş Deneyimi (Work experience) Bölümünü Çek ---
    work_header = soup.find("h2", string=lambda t: t and "Work experience" in t)
    if work_header:
        current_tag = work_header.find_next_sibling()
        while current_tag and current_tag.name != 'h2':
            if current_tag.name == 'p' or current_tag.name == 'h4':
                cv_text_list.append(current_tag.text.strip())
            current_tag = current_tag.find_next_sibling()

    # --- 3. Beceri (Skills) Bölümünü Çek ---
    skills_header = soup.find("h2", string=lambda t: t and "Skills" in t)
    if skills_header:
        
        # --- YENİ GÜNCELLEME (HATANIN ÇÖZÜMÜ) ---
        # 'find_next_sibling("ul")' yerine, daha esnek olan 'find_next("ul")' kullanıyoruz.
        # Bu, <h2> ile <ul> arasında başka etiketler olsa bile listeyi bulur.
        skills_ul = skills_header.find_next("ul") 
        # ---------------------------------------------
        
        if skills_ul:
            skill_items = skills_ul.find_all("li")
            for item in skill_items:
                beceri_metni = item.text.strip()
                if beceri_metni: # Boş <li> etiketlerini atla
                    cv_text_list.append(beceri_metni)
                    cv_beceri_listesi.append(beceri_metni)

    # --- 4. Sertifika (Certificates) Bölümünü Çek ---
    cert_header = soup.find("h2", string=lambda t: t and "Certificates" in t)
    if cert_header:
        current_tag = cert_header.find_next_sibling()
        while current_tag and current_tag.name != 'h2':
            if current_tag.name == 'p':
                cv_text_list.append(current_tag.text.strip())
            current_tag = current_tag.find_next_sibling()
            
    # --- 5. Çıktıları Birleştir ---
    cv_full_text = "\n".join(list(filter(None, cv_text_list))) # Boş satırları kaldır
    
    df_cv_skills = pd.DataFrame(cv_beceri_listesi, columns=["ham_beceri_adi"])
    df_cv_skills.to_pickle("IP1_Ciktisi_Aday_Ham_Beceriler.pkl")
    
    with open("IP1_Ciktisi_Aday_Tam_Metin.txt", "w", encoding="utf-8") as f:
        f.write(cv_full_text)

    print(f"\n--- İP 1 - Adım 3 TAMAMLANDI (SELENIUM/DÜZELTİLMİŞ) ---")
    print("Europass CV (HTML) başarıyla ayrıştırıldı.")
    print(f"Toplam {len(cv_beceri_listesi)} adet ham beceri metni bulundu ve 'IP1_Ciktisi_Aday_Ham_Beceriler.pkl' dosyasına kaydedildi.")
    print("CV'nin tam metni 'IP1_Ciktisi_Aday_Tam_Metin.txt' dosyasına kaydedildi.")

except requests.exceptions.RequestException as e:
    print(f"HATA: Europass linkine bağlanılamadı: {e}")
except Exception as e:
    print(f"Bilinmeyen bir hata oluştu: {e}")
finally:
    driver.quit()
    print("Tarayıcı kapatıldı.")

İP 1 - Adım 3 Başladı: Europass CV linki Selenium ile çekiliyor...
URL: https://europa.eu/europass/eportfolio/api/eprofile/shared-profile/aziz+muammer-%C3%A7etiner/7166a565-f20c-4c5c-83a4-00a4db71fd09?view=html
Europass profili (JavaScript) yüklendi. Veri ayrıştırılıyor...

--- İP 1 - Adım 3 TAMAMLANDI (SELENIUM/DÜZELTİLMİŞ) ---
Europass CV (HTML) başarıyla ayrıştırıldı.
Toplam 14 adet ham beceri metni bulundu ve 'IP1_Ciktisi_Aday_Ham_Beceriler.pkl' dosyasına kaydedildi.
CV'nin tam metni 'IP1_Ciktisi_Aday_Tam_Metin.txt' dosyasına kaydedildi.
Tarayıcı kapatıldı.
